In [101]:
import xml.etree.ElementTree as ET
import numpy as np


cco_file = '../tree_3D.xml'
output_file = '../examples/cube__n3k_a10k.vtk'

tree = ET.parse(cco_file)
root = tree.getroot()

nodes = []  # to store node positions
edges = []  # to store edge indices and associated data
flows = []  # to store flow values
resistances = []  # to store resistance values
radii = []  # to store radius values

# Extract node positions
for node in root.findall('.//node'):
    position = [float(elem.text) for elem in node.findall('attr[@name=" position"]/tup/float')]
    nodes.append(position)

# Extract edge data, flow values, resistance values and radius values
for edge in root.findall('.//edge'):
    from_node = edge.get('from').replace('n', '')  # Get index of "from" node
    to_node = edge.get('to').replace('n', '')  # Get index of "to" node
    
    flow = float(edge.find("attr[@name=' flow']/float").text)  # Get flow value
    resistance = float(edge.find("attr[@name=' resistance']/float").text)  # Get resistance value
    radius = float(edge.find("attr[@name=' radius']/float").text)  # Get radius value
    
    edges.append([int(from_node), int(to_node)])
    flows.append(flow)
    resistances.append(resistance)
    radii.append(radius)

In [102]:
edges = np.array(edges)
flows = np.array(flows)
resistances = np.array(resistances)
radii = np.array(radii)

In [103]:

def write_vtk(filename, vertices, edges, dict = {}):
    """ Write a VTK file """
    f = open(filename, "w")

    f.write("# vtk DataFile Version 3.0\n")
    f.write("Fibers output - vtk\n")
    f.write("ASCII\n")
    f.write("DATASET UNSTRUCTURED_GRID\n")
    f.write("POINTS " + str(len(vertices)) + " DOUBLE\n")

    for v in vertices:
        f.write(str(v[0]) + " " + str(v[1]) + " " + str(v[2]) + "\n")

    f.write("CELLS " + str(len(edges)) + " " + str(3 * len(edges)) + "\n")
    for e in edges:
        f.write("2 " + str(int(e[0])) + " " + str(int(e[1])) + "\n")

    f.write("CELL_TYPES " + str(len(edges)) + "\n")
    for e in edges:
        f.write("3\n")
    
    # Check if we have point or cell data
    for key, data in dict.items():
      if len(data) == len(vertices):
        f.write("POINT_DATA " + str(len(data)) + "\n")
      elif len(data) == len(edges):
        f.write("CELL_DATA " + str(len(data)) + "\n")
      else:
        return 
      f.write("SCALARS " + key + " float \n")
      f.write("LOOKUP_TABLE default \n")
      for l in data:
          f.write(str(l) + "\n")
      
    f.close()

In [104]:
write_vtk(output_file, nodes, edges, {'radius':radii})

In [105]:
nodes = np.array(nodes)
fluid_volume = 0
weighted_length = 0
weigthed_direction = [0,0,0]
for e,r in zip(edges,radii):
    length = np.linalg.norm(nodes[e[1]] - nodes[e[0]])
    weighted_length += length*r
    weigthed_direction += r*(nodes[e[1]] - nodes[e[0]])
    fluid_volume += length*r**2*np.pi

weighted_length = weighted_length/len(edges)
weigthed_direction = weigthed_direction/len(edges)


In [106]:
fluid_volume/33**3

np.float64(0.01748422556187042)

In [107]:
np.linalg.norm(nodes[e[1]] - nodes[e[0]])


np.float64(0.8224305310480643)

In [108]:
print(" ** number of nodes: ", nodes.shape[0])
print(" ** fluid volume ratio: ", fluid_volume/26.7**3)
print(" ** number of vessels: ", edges.shape[0])
print(" ** weighted average length: ", weighted_length)
print(" ** weighted average direction: ", weigthed_direction)

 ** number of nodes:  6000
 ** fluid volume ratio:  0.03301067738134518
 ** number of vessels:  5999
 ** weighted average length:  0.15029727020564956
 ** weighted average direction:  [ 0.00255979 -0.02805087  0.00174366]
